In [6]:
import sys
from pyspark.sql import functions as F
from spark_session import get_spark

In [7]:
from quality_py import run_checks
import sources

In [8]:
%run cleaning.ipynb

In [9]:
LAKE, W = "data/lake/orders_current", "data/warehouse"

In [10]:
spark = get_spark("etl")

In [ ]:
def dim_customer(spark):
    guest = spark.createDataFrame([(0, "Guest", "Unknown", None)], "customer_id INT, name STRING, city STRING, signup_date DATE")
    return clean_customers(sources.customers(spark)).unionByName(guest)


def dim_date(orders):
    span = orders.agg(F.min("order_date").alias("a"), F.max("order_date").alias("b"))
    return (span.select(F.explode(F.sequence("a", "b")).alias("date"))
            .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
            .withColumn("day_name", F.date_format("date", "EEEE"))
            .withColumn("is_weekend", F.dayofweek("date").isin(1, 7))
            )
    

def fact_sales(orders, products):
    revenue = F.when(F.col("status") == "refunded", 0.0).otherwise(F.round(F.col("quantity") * F.col("price"), 2))
    return (orders.join(F.broadcast(products.select("product_id", "price")), "product_id")
            .withColumn("date_key", F.date_format("order_date", "yyyyMMdd").cast("int"))
            .withColumn("revenue", revenue)
            .select("order_id", "date_key", "customer_id", "product_id", "quantity", F.col("price").alias("unit_price"), "revenue", "status"))


def publish(df, name, partition=None):
    w = df.write.mode("overwrite")
    if partition:
        w = w.partitionBy(partition)
    w.parquet(f"{W}/{name}")
    print(f"published {name}")

In [12]:
orders = spark.read.parquet(LAKE).cache()
products = sources.products(spark)
if not run_checks(orders, products):
    sys.exit("quality gate failed: nothing published")
customers = dim_customer(spark)
fact = fact_sales(orders, products).cache()
orphans = fact.join(customers, "customer_id", "left_anti").count()
if orphans:
    sys.exit(f"{orphans} fact rows without a customer")
publish(customers, "dim_customer")
publish(products, "dim_product")
publish(dim_date(orders), "dim_date")
publish(fact, "fact_sales", partition="date_key")
print("fact rows:", fact.count(), "revenue:", fact.agg(F.round(F.sum("revenue"), 2)).first()[0])

PASS not empty: 309032 rows
PASS unique order_id: 0 duplicates
PASS required not null: 0 rows
PASS quantity positive: 0 rows
PASS known status: 0 rows
PASS product exists: 0 rows
latest orders:  2026-10-17 07:59:51


PySparkTypeError: [FIELD_DATA_TYPE_UNACCEPTABLE] StructType([StructField('customer_id', IntegerType(), True), StructField('name', StringType(), True), StructField('city', StringType(), True), StructField('signup_date', DateType(), True)]) can not accept object 0 in type <class 'int'>.